In [1]:
import pyrosm
import pandas as pd
import geopandas as gpd
import urllib.request
import warnings
from os.path import exists
import osmnx as ox
import numpy as np
from shapely import wkt
from fmm import Network, NetworkGraph, STMATCH, STMATCHConfig
from shapely.geometry import LineString
warnings.filterwarnings("ignore")

/home/napo/.local/lib/python3.9/site-packages/geopandas/_compat.py:111: UserWarning: The Shapely GEOS version (3.9.1-CAPI-1.14.2) is incompatible with the GEOS version PyGEOS was compiled with (3.10.1-CAPI-1.16.0). Conversions between both will be slow.
  warnings.warn(


In [2]:
trips = pd.read_parquet('../data/trips_pointv3.parquet')

In [3]:
new_columns = {}
for n in list(trips.columns):
    new_columns[n] = n.replace('point_','')

In [4]:
trips.rename(columns=new_columns,inplace=True)

In [5]:
trips.longitude = trips.longitude.astype(float)
trips.latitude = trips.latitude.astype(float)

In [6]:
geo_trips = gpd.GeoDataFrame(
    trips,
    crs=4326,
    geometry=gpd.points_from_xy(trips.longitude, trips.latitude))

In [7]:
geo_trips.trip_id.unique().size

6475

In [8]:
trips_id = geo_trips.groupby('trip_id').size().to_frame().reset_index()

In [9]:
trips_id['total'] = trips_id[0]

In [10]:
del trips_id[0]


In [11]:
geo_trips.sort_values(['trip_id','timestamp'],inplace=True)

In [12]:
if(exists("../data/trips_streets.gpkg")):
    trips_streets = gpd.read_file("trips_streets.gpkg")
else:
    trips_id = geo_trips[['trip_id']]
    trips_id['geometry'] = trips_id.trip_id.apply(lambda x: LineString(
        geo_trips[geo_trips.trip_id == x].geometry.to_list()))
    trips_id['duration'] = trips_id.trip_id.apply(
        lambda x: max(geo_trips[geo_trips.trip_id == x].timestamp) -
        min(geo_trips[geo_trips.trip_id == x].timestamp))
    trips_id['duration'] = trips_id.duration.apply(lambda x: x.total_seconds())
    trips_id['start_day'] = trips_id.trip_id.apply(
        lambda x: min(geo_trips[geo_trips.trip_id == x].timestamp))
    trips_streets = gpd.GeoDataFrame(
        trips_id,
        crs=4326,
        geometry=trips_id.geometry)
    trips_streets.drop_duplicates(subset=None, keep='first', inplace=True)
    trips_streets.to_file("trips_streets.gpkg", driver="GPKG")


In [13]:
url_download_trento_pbf = "https://osmit-estratti-test.wmcloud.org/dati/poly/comuni/pbf/022205_Trento_poly.osm.pbf"
pbf_filename = "trento_osm.pbf"
if (exists("trento_osm.pbf") == False):
    urllib.request.urlretrieve(url_download_trento_pbf, pbf_filename)   

In [14]:
#bounding_box=trips_streets.geometry.unary_union.convex_hull

In [15]:
osm = pyrosm.OSM(pbf_filename) #,bounding_box=bounding_box)

In [16]:
# creation of the graph
nodes, edges = osm.get_network(network_type="cycling", nodes=True)
G = osm.to_graph(nodes, edges, graph_type="networkx")
%time

CPU times: user 5 µs, sys: 1 µs, total: 6 µs
Wall time: 10.5 µs


In [17]:
# archive edges and nodes in ESRI Shapefile (required of FMM)
encoding = "utf-8"
filepath_nodes = "nodes.shp"
filepath_edges = "edges.shp"
if (exists("nodes.shp") == False):
    gdf_nodes, gdf_edges = ox.utils_graph.graph_to_gdfs(G)
    # convert undirected graph to gdfs and stringify non-numeric columns
    gdf_nodes = ox.io._stringify_nonnumeric_cols(gdf_nodes)
    gdf_edges = ox.io._stringify_nonnumeric_cols(gdf_edges)
    gdf_edges["fid"] = np.arange(0, gdf_edges.shape[0], dtype='int')
    # save the nodes and edges as separate ESRI shapefiles
    del gdf_nodes['osmid']
    gdf_nodes.to_file(filepath_nodes, encoding=encoding)
    gdf_edges.to_file(filepath_edges, encoding=encoding)

In [18]:
network = Network("edges.shp", "fid", "u", "v")
graph = NetworkGraph(network)

In [19]:
model = STMATCH(network,graph)
k = 1  # number of candidates
gps_error = 0.0005  # GPS error, unit is map_unit.
radius = 0.003  # search radius for candidates, unit is map_unit
vmax = 30  # maximum speed of the vehicle, unit is map_unit/second
stmatch_config = STMATCHConfig(k, radius, gps_error, vmax)


In [20]:
def getMapMatching(model, stmatch_config, route_wkt):
    result = model.match_wkt(route_wkt, stmatch_config)
    matched_path = list(result.cpath)
    matched_edge_for_each_point = list(result.opath)
    matched_edge_index = list(result.indices)
    linestring_wkt = result.mgeom.export_wkt()
    point_wkt = result.pgeom.export_wkt()
    return (matched_path, matched_edge_for_each_point, matched_edge_index, linestring_wkt, point_wkt)


In [21]:
trips_streets['result'] = trips_streets['geometry'].apply(
    lambda x: getMapMatching(model, stmatch_config, x.wkt))


In [22]:
trips_streets['matched_path'] = trips_streets['result'].apply(lambda x: x[0])
trips_streets['matched_edge_for_each_point'] = trips_streets['result'].apply(
    lambda x: x[1])
trips_streets['matched_edge_index'] = trips_streets['result'].apply(
    lambda x: x[2])
trips_streets['linestring_wkt'] = trips_streets['result'].apply(lambda x: x[3])
trips_streets['point_wkt'] = trips_streets['result'].apply(lambda x: x[4])


In [23]:
del trips_streets['result']


In [24]:
trips_streets['geometry_matched'] = trips_streets['geometry'].apply(lambda x: wkt.loads(x.wkt))
  